# Mikołaj: RAG (Część II)

Zakres (3 pkt): Część II z [instrukcji](../Zadanie_3_zmodyfikowane.md). Ta część **nie zależy od Części I**: na start wystarczy własny indeks FAISS. Retriever hybrydowy z Części I (pkt 2.b) można podpiąć później z `../utils/`.

## Do zrobienia

1. **II.A: artykuł o RAG** (Medium, RAG basics). Krótka notatka w notebooku.
2. **Pkt 1: API CLARIN**: rejestracja na [services.clarin-pl.eu](https://services.clarin-pl.eu/login), lista modeli (`/api/v1/oapi/models`), uruchomienie modelu (`/api/v1/oapi/chat/completions`). Klucz API trzymać poza repo (zmienna środowiskowa).
3. **Pkt 2: dane i indeks**: wybrać zbiór (np. IMDB `stanfordnlp/imdb`) i zaindeksować go (lub użyć gotowego indeksu FAISS Wikipedii z Kaggle).
4. **Pkt 3: moduł generacji (RAG)**:
   - a. prompt zamieniający znalezione dokumenty na odpowiedź w języku naturalnym,
   - b. własne przykładowe zapytania przepuszczone przez retriever i moduł RAG,
   - c. analiza rezultatów po zastosowaniu RAG,
   - d. podsumowanie wyników.

In [ ]:
import sys; sys.path.insert(0, "..")

## II.A: notatka z artykułu

RAG (Retrieval-Augmented Generation) łączy dwa elementy: wyszukiwarkę (retriever) i model językowy (generator). Najpierw dokumenty dzieli się na fragmenty, zamienia na wektory (osadzenia) i zapisuje w bazie wektorowej, np. FAISS. Przy pytaniu retriever zamienia je na wektor i znajduje najbardziej podobne fragmenty. Te fragmenty trafiają do promptu razem z pytaniem, a model odpowiada na ich podstawie. Po co: model dostaje aktualną lub prywatną wiedzę spoza swoich wag, rzadziej zmyśla (halucynuje) i można wskazać źródło odpowiedzi. Prompt jest kluczowy, bo mówi modelowi, żeby używał tylko podanego kontekstu. Ograniczenia: jakość odpowiedzi zależy od retrievera (zły fragment daje złą odpowiedź), a liczba fragmentów jest ograniczona długością kontekstu modelu.

## 1. Model językowy CLARIN

Token jest w `03/.env` (wzór: `03/.env.example`), czytany przez `pydantic-settings`.

In [ ]:
import random

import pandas as pd

from utils.rag.data import load_squad
from utils.rag.retriever import FaissRetriever
from utils.rag.llm import list_models, get_llm
from utils.rag.pipeline import build_rag_chain, build_plain_chain, contains_gold

pd.set_option("display.max_colwidth", None)
list_models()

**Wybór modelu:** _(uzupełnić po obejrzeniu listy: nazwa modelu, rozmiar, obsługa języka angielskiego, dostępność; wpisać nazwę do `CLARIN_MODEL` w `03/.env`)_

In [ ]:
llm = get_llm()
llm.invoke("Say hello in Polish.").content

## 2. Indeks i wyszukiwanie

Akapity SQuAD (split `validation`) osadzone modelem `multi-qa-mpnet-base-dot-v1`, indeks FAISS z iloczynem skalarnym. Osadzenia są zapisywane w `03/artifacts/rag/`, więc kolejne uruchomienie jest szybkie.

In [ ]:
docs, qa = load_squad()
retriever = FaissRetriever(docs)

In [ ]:
search_queries = ["Who won Super Bowl 50?", "What is the capital of Poland?", "How do plants make energy?"]
pd.DataFrame(
    [
        {"query": q, "rank": r, "score": round(d.score, 1), "title": d.title, "text": d.text[:100]}
        for q in search_queries
        for r, d in enumerate(retriever.retrieve(q, k=3), 1)
    ]
)

_(uzupełnić po uruchomieniu: czy najwyższe wyniki dotyczą właściwych artykułów; co dzieje się dla pytania spoza bazy, np. o rośliny)_

## 3. Łańcuchy RAG i bez RAG

`rag_chain`: pytanie → retriever (`k = 5`) → prompt z numerowanymi akapitami → model → `{question, docs, answer}`. `plain_chain`: samo pytanie do modelu, bez kontekstu.

In [ ]:
rag_chain = build_rag_chain(retriever, llm, k=5)
plain_chain = build_plain_chain(llm)

In [ ]:
# Article titles in the index, to check which own questions are covered by the base.
sorted({d.title for d in docs})

Po obejrzeniu listy tytułów **dostosować pytania własne** tak, żeby dwa były z tematów obecnych w bazie, a dwa spoza bazy (wtedy oczekiwane jest `I don't know.`). Pytanie z fałszywym założeniem sprawdza, czy model da się podpuścić.

In [ ]:
questions = [
    # from SQuAD
    qa[0].question,
    qa[100].question,
    qa[2000].question,
    # own, topic in the base
    "Which city hosted Super Bowl 50?",
    "What did Martin Luther publish in 1517?",
    # own, topic outside the base
    "What is the population of Wrocław?",
    "Who wrote the novel Pan Tadeusz?",
    # false premise
    "Who invented the steam engine in Poland?",
]
rows = []
for q in questions:
    out = rag_chain.invoke(q)
    rows.append(
        {
            "question": q,
            "plain": plain_chain.invoke({"question": q}),
            "rag": out["answer"],
            "sources": [d.title for d in out["docs"]],
        }
    )
pd.DataFrame(rows)

_(uzupełnić po uruchomieniu: 2–3 obserwacje z tabeli, np. czy RAG odpowiada `I don't know.` dla pytań spoza bazy, a model bez RAG zgaduje)_

## 4. Miara liczbowa (50 pytań)

Losowa próbka 50 pytań z SQuAD (ziarno 42). Cztery liczby:

- trafność retrievera: czy akapit z odpowiedzią jest w top 5 (hit@5),
- RAG: czy odpowiedź zawiera wzorzec (`contains_gold`),
- bez RAG: to samo dla modelu bez kontekstu,
- liczba odpowiedzi `I don't know.` z RAG.

Przy błędach HTTP 429 (limit zapytań) zmniejszyć `max_concurrency` do `1`.

In [ ]:
N = 50
sample = random.Random(42).sample(qa, N)
qs = [s.question for s in sample]
rag_out = rag_chain.batch(qs, config={"max_concurrency": 4})
plain_out = plain_chain.batch([{"question": q} for q in qs], config={"max_concurrency": 4})

counts = {
    "retriever hit@5": sum(s.doc_id in {d.id for d in o["docs"]} for s, o in zip(sample, rag_out)),
    "RAG contains gold": sum(contains_gold(o["answer"], s.answers) for s, o in zip(sample, rag_out)),
    "no-RAG contains gold": sum(contains_gold(p, s.answers) for s, p in zip(sample, plain_out)),
    "RAG \"I don't know.\"": sum(o["answer"].strip() == "I don't know." for o in rag_out),
}
pd.DataFrame({"count": counts, "share": {k: v / N for k, v in counts.items()}})

## 5. Przykłady błędów

Pierwsze 6 pytań, na które odpowiedź RAG nie zawiera wzorca, razem z informacją, czy właściwy akapit został znaleziony.

In [ ]:
by_id = {d.id: d for d in docs}
misses = [(s, o) for s, o in zip(sample, rag_out) if not contains_gold(o["answer"], s.answers)]
pd.DataFrame(
    [
        {
            "question": s.question,
            "gold": s.answers,
            "rag": o["answer"],
            "right paragraph retrieved": s.doc_id in {d.id for d in o["docs"]},
            "right article": by_id[s.doc_id].title,
            "retrieved": [d.title for d in o["docs"]],
        }
        for s, o in misses[:6]
    ]
)

## Analiza

Błędy RAG z sekcji 4 dzielę na cztery kategorie:

- **(a) Retriever nie znalazł akapitu.** Właściwego akapitu nie ma w top 5, więc model nie ma z czego odpowiedzieć. Liczba: _(uzupełnić po uruchomieniu)_.
- **(b) Akapit był, ale odpowiedź inna lub szersza niż wzorzec.** To często fałszywy błąd miary: `contains_gold` wymaga dosłownego wzorca, a model mógł podać poprawną parafrazę. Liczba: _(uzupełnić po uruchomieniu)_.
- **(c) Halucynacja.** Akapit był, a odpowiedź jest błędna merytorycznie lub nie ma pokrycia w kontekście. Liczba: _(uzupełnić po uruchomieniu)_.
- **(d) `I don't know.` mimo dobrego akapitu.** Prompt jest zbyt ostrożny albo model nie znalazł odpowiedzi w długim kontekście. Liczba: _(uzupełnić po uruchomieniu)_.

Pytania spoza bazy: _(uzupełnić po uruchomieniu: co odpowiedział RAG, a co model bez RAG)_.

Porównanie z modelem bez RAG: _(uzupełnić po uruchomieniu: różnica `RAG contains gold` i `no-RAG contains gold`, w których pytaniach RAG pomógł, a w których zaszkodził)_.

## Podsumowanie

_(uzupełnić po uruchomieniu: 3–5 zdań: ile daje RAG względem modelu bez kontekstu, jak duży udział mają błędy retrievera, jak model zachowuje się poza bazą, co poprawić, np. większe `k`, lepszy retriever hybrydowy z Części I)_